# Currency Exchange ETL Pipeline

This notebook orchestrates an ETL (Extract, Transform, Load) pipeline to fetch historical currency exchange rates from the Frankfurter API. The pipeline is designed to be idempotent and robust, utilizing **Prefect** for orchestration and **SQLite** for local storage.

In [ ]:
import requests
import pandas as pd
import sqlite3
from prefect import task, flow

## 1. Data Extraction

The extraction phase connects to the external API to retrieve the raw financial data. We implement retry mechanisms using Prefect's task decorators to ensure resilience against temporary network instability or API rate limits.

Función de extraccion de los datos

In [ ]:
@task(retries=3, retry_delay_seconds=10)
def extract_data(url="https://api.frankfurter.dev/v2/rates", 
                 headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}, 
                 date_range=["2010-01-01", "2025-12-31"], 
                 currencies="usd,gbp", 
                 base="eur"):
    """
    Extracts exchange rate data from the Frankfurter API.
    """
    api_url = f"{url}?base={base}&quotes={currencies}&from={date_range[0]}&to={date_range[1]}"
    response = requests.get(api_url, headers=headers)

    if response.status_code == 200:
        print("Data successfully extracted from API.")
        return response.json()
    else:
        print(f"Connection error. Status code: {response.status_code}")
        return None

## 2. Data Transformation

Once the raw JSON data is acquired, it must be transformed into a structured tabular format. This step parses the nested payload into a Pandas DataFrame and converts date strings into native datetime objects for accurate time-series analysis downstream.

In [ ]:
@task
def clean_data(raw_data):
    """
    Transforms raw JSON data into a cleaned Pandas DataFrame.
    """
    df = pd.DataFrame(raw_data)
    df['date'] = pd.to_datetime(df['date'])
    
    print("Data successfully cleaned and formatted.")
    return df

## 3. Data Loading (Idempotent)

The transformed data is persisted into a local SQLite database. To maintain idempotency and prevent duplicate records upon consecutive runs, the process dynamically identifies the date range and currencies in the current batch. It performs a targeted deletion of overlapping historical entries before appending the new data.

In [ ]:
@task
def load_data(df, db_name="historico_divisas.db", table_name="tasas_cambio"):
    """
    Loads the cleaned DataFrame into a SQLite database.
    Implements a Delete & Append pattern to prevent duplicates.
    """
    try:
        conn = sqlite3.connect(db_name)
        cursor = conn.cursor()

        # Extract date extremes as strings for SQLite compatibility
        min_date = str(df['date'].min())
        max_date = str(df['date'].max())
        
        # Identify unique currencies present in the current batch
        new_currencies = df['quote'].unique().tolist()
        
        # Generate placeholders to build a dynamic and secure SQL query
        placeholders = ", ".join(["?"] * len(new_currencies))
        
        # Build and execute the dynamic delete query to ensure idempotency
        delete_query = f"DELETE FROM {table_name} WHERE date >= ? AND date <= ? AND quote IN ({placeholders})"
        params = [min_date, max_date] + new_currencies
        
        cursor.execute(delete_query, params)
        conn.commit()

        # Append new data safely without duplicating existing records
        df.to_sql(table_name, conn, if_exists='append', index=False)
        
        print(f"Load successful: {len(df)} rows inserted into '{table_name}' (No duplicates).")
        
    except Exception as e:
        print(f"Database loading error: {e}")
        
    finally:
        conn.close()

## 4. Pipeline Orchestration

Prefect manages the execution graph. The main flow function ties the extraction, cleaning, and loading tasks together, providing a parameterized entry point for the pipeline.

In [ ]:
@flow(name="Currency-ETL-Pipeline") 
def currency_etl_flow(url="https://api.frankfurter.dev/v2/rates", 
                      date_range=["2010-01-01", "2025-12-31"], 
                      currencies="usd,gbp,jpy,cny,aud,cad", 
                      base="eur"):
    """
    Main Prefect flow orchestrating the ETL tasks.
    """
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    
    raw_data = extract_data(url=url, headers=headers, date_range=date_range, currencies=currencies, base=base)
    
    # Defensive check: proceed only if extraction was successful
    if raw_data:
        clean_df = clean_data(raw_data)
        load_data(clean_df)
    else:
        print("Pipeline stopped: No data extracted.")

if __name__ == "__main__":
    currency_etl_flow()

In [ ]:
main_etl(currencies="usd,gbp,jpy,cny,aud,cad", range_time=["2020-01-01", "2025-12-31"])

13:28:34.164 | INFO    | Flow run 'educational-seal' - Beginning flow run 'educational-seal' for flow 'PipeLine-Divisas'

Código de estado devuelto: 200
Datos extraidos con exito


13:28:34.407 | INFO    | Task run 'extract_data-459' - Finished in state Completed()

Datos limpiados con exito


13:28:35.680 | INFO    | Task run 'clean_data-fb3' - Finished in state Completed()

Carga exitosa: 35064 filas insertadas en la tabla 'tasas_cambio'.


13:28:35.746 | INFO    | Task run 'load_data-2eb' - Finished in state Completed()

13:28:36.174 | INFO    | Flow run 'educational-seal' - Finished in state Completed()